# Cluster Sentences: Find communities of sentences that are highly similar to each other.

This notebook looks into techniques for defining clusters and their visualisation.
- Same sentence selection was used that was used for making the graph.
- Identify clusters with 'fast clustering' based on cosine similarity. (= `community_SBERT`)
- Visualise clusters with PCA, UMAP, and graph.
- TODO: wordclouds

### Settings

In [ ]:
# Experiment name
experiment_name = "free_3600_250606"
graph_file = f"graph_thres_0_75_{experiment_name}.pickle"

# Thresholds
community_threshold = 0.75
min_community_size = 50

# For consistent graph plotting
seed = 42

## Initalisation

### Imports

In [ ]:
import pickle
import pandas as pd
import numpy as np
from sentence_transformers import util

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from umap import UMAP
import networkx as nx

### Functions

In [ ]:
# For normalizing the vectors:
# function of getting magnitude/length/norm
def get_norm(matrix):
    """Calculate the norm of a vector"""
    norm = np.sqrt(
        np.sum(
            np.power(
                matrix, 
                2
            ),
            axis=1
        )
    )

    return norm

# function to divide vector by norm
# (..probably there is a numpy function for it.. but i failed to find it)
def normalize(matrix, scales):
    """Divide elements in matrix by given scale"""
    return np.array([vector / scales[idx] for idx, vector in enumerate(matrix)])

In [ ]:
def plot_clusters(embed, df_vector_labels, column):
    # get unique colours
    colours = sns.color_palette("nipy_spectral", df_vector_labels[column].nunique())

    for idx, val in enumerate(sorted(df_vector_labels[column].unique())):
        projection = embed[df_vector_labels[column] == val]
        plt.scatter(projection.T[0], projection.T[1], alpha=0.1, c=colours[idx], label= val)
    plt.title(f"Coloured by {column}")

In [ ]:
def plot_graph(G, pos, vector_labels: pd.DataFrame, filter_column: str,):
    """Plot graph `G` on `pos`itions, and colors clusters by looking for node_ids based on the cluster_id in given vector_labels dataframe."""
    nx.draw_networkx_edges(G, pos, width=1, alpha= 0.1)

    # get unique colours for the unique cluster_ids
    colours = sns.color_palette("nipy_spectral", vector_labels[filter_column].nunique())

    # for cluster_id in vector_labels dataframe
    for idx, cluster_id in enumerate(np.sort(vector_labels[filter_column].unique())):
        # get list of nodes
        nodelist = vector_labels.loc[vector_labels[filter_column] == cluster_id].index.to_list()

        # if cluster_id == -1, then it is not within a called cluster, and will be colored gray.
        if cluster_id == -1:
            nx.draw_networkx_nodes(
                G,
                pos,
                nodelist= nodelist,
                node_color= "tab:gray",
                edgecolors= "tab:gray",
                node_size= 40,
                alpha= 0.1
            )

            # Empty scatter is a trick to add a legend to graph (https://stackoverflow.com/questions/60706947/how-to-add-a-legend-to-networkx-graph-based-on-node-colour)
            plt.scatter([],[], c="tab:gray", label= f'Cluster {cluster_id}')

        # color each node within the same cluster with same color. Different clusters will have different colours
        else:
            nx.draw_networkx_nodes(
                G,
                pos,
                nodelist= nodelist,
                node_color= colours[idx],
                node_size= 40,
                alpha= 0.4
            )

            # Empty scatter is a trick to add a legend to graph (https://stackoverflow.com/questions/60706947/how-to-add-a-legend-to-networkx-graph-based-on-node-colour)
            plt.scatter(
                [],[],
                c= colours[idx],
                label= f'Cluster {cluster_id}'
            )

    # Add legend
    plt.legend(ncol=3,bbox_to_anchor= (2.3, 1))
    plt.tight_layout()

    plt.axis("off")

### Load files

In [ ]:
# Filenames
embedding_file = f"embedding_{experiment_name}.pickle"
raw_corpus_file = f"corpus_{experiment_name}.csv"
meta_file = f'meta_{experiment_name}.csv'

# Load graph
with open(f"../../data/graphs/{graph_file}", 'rb') as handle:
    G = pickle.load(handle)

# Load raw corpus
df_corpus = pd.read_csv(f'../../data/corpus/{raw_corpus_file}')

# Load metadata
df_meta = pd.read_csv(f'../../data/meta/{meta_file}', index_col=0)

# Load sentence embeddings
with open(f"../../data/vectors/{embedding_file}", 'rb') as handle:
    embeddings = pickle.load(handle)

# Round to 4 decimals to compensate for float point errors.
embeddings = np.round(embeddings, decimals=4)

In [ ]:
# Check if embeddings normalized,
# if not, it will be normalized:
if np.sum(get_norm(embeddings)) == embeddings.shape[0]:
    print("Embedding is already normalized. (`normalized_embeddings == embeddings`)")
    normalized_embeddings = embeddings

else:
    print("Embedding is not normalized yet.")
    print("Applying normalization.")
    normalized_embeddings = normalize(
        embeddings,
        get_norm(
            embeddings
        )
    )
    print("Normalized embedding successful created. (`normalized_embeddings != embeddings`)")

## Pre-processing

### Filter on only sentences that are in the graph

In [ ]:
# Get indexes of sentences of interest
soi_idx = sorted(G.nodes)

embeddings = pd.DataFrame(embeddings).loc[soi_idx]
normalized_embeddings = pd.DataFrame(normalized_embeddings).loc[soi_idx]

### Get sentence_id dictionary

The cluster finding algorithm does not take the sentence_id, but will return the 'enumarate_id' instead. Therefor a dictionary is created to retrieve the sentence_id.

In [ ]:
# Get sentence_ids and enumerate_ids
sentence_ids = embeddings.index
enumerate_ids = np.arange(embeddings.shape[0])

# Get dict where enumerate_ids can be transformed to sentence_ids
enum_sent_dict = dict(zip(enumerate_ids, sentence_ids))

### Prepare metadata

If the code below shows results.. than see next comment below

In [ ]:
df_meta[df_meta.pmid.duplicated()]

In [ ]:
df_meta[df_meta.pmid == 34792911]

NOTE: I noted that some pmids are entered twice in the meta data csv file.............. duplicate of pmid [15551110, 34792911] have been removed from the meta data file. (original is kept under `meta_*_ori.csv`)

In [ ]:
# Replace " PD " with " parkinson disease ". (Important for generating a wordcloud)
df_corpus["sentence_text"] = df_corpus["sentence_text"].str.replace(
    "PD",
    "parkinson's disease",
    case= True
)

In [ ]:
# Join metadata of paper
df_corpus.rename(columns={'paper_name': 'pmid'}, inplace=True)

df_vector_labels = df_corpus.join(
    df_meta[['pmid', 'is_accepted', 'is_published', 'is_retracted', 'pub_year', 'cited_by_count', 'referenced_count']].set_index('pmid'),
    on='pmid',
    how='left'
)
df_vector_labels.tail()

## Clustering

In [ ]:
# NOTE: threshold of 0.75 is based on graph in 'sentence_pairs.ipynb', section '### Distribution of highest similarity scores'
enum_idx_clusters = util.community_detection(
    embeddings.to_numpy(), 
    min_community_size= min_community_size, 
    threshold= community_threshold
)

# Change enumerate_id to sentence_id
sent_idx_clusters = [[enum_sent_dict[enum_id] for enum_id in comm_list] for comm_list in enum_idx_clusters]

In [ ]:
all_sentences = [sent for clique in sent_idx_clusters for sent in clique]
print(f"Number of clusters found: {len(sent_idx_clusters)}")
print(f"{round(len(all_sentences)/embeddings.shape[0] * 100, 2)}% ({len(all_sentences)}/{embeddings.shape[0]}) sentences are placed in a cluster.")

#### Add column `community` to df_vector_labels

In [ ]:
# Add column with `cluster_id` (cluster_id == -1 when it's not assigned to a clusters)
# NOTE: The key in this case is the index of the list, since the index may have changed when removing duplicates.
df_vector_labels["community_SBERT"] = -1
for idx, clique in enumerate(sent_idx_clusters):

    # clique = [comm_vector_dict[comm_idx] for comm_idx in clique]
    df_vector_labels.loc[df_vector_labels.index.isin(clique), 'community_SBERT'] = idx

In [ ]:
df_vector_labels.head()

## Visualisation

### Plot only sentences with a similarity score >= 0.75 (PCA, UMAP, Graph)

#### PCA

In [ ]:
# Apply PCA
pca = PCA(n_components=2)
pca_embeddings = pca.fit_transform(normalized_embeddings)
# pca_embeddings = pca.fit_transform(embeddings)
pca_embeddings[:5]

In [ ]:
plot_clusters(
    pca_embeddings, 
    df_vector_labels.loc[enum_sent_dict.values()], 
    'community_SBERT'
    # 'pub_year'
)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)")
plt.show()

#### UMAP

In [ ]:
# NOTE: Takes some time... (~1 minute)

umap = UMAP(
        n_components= 2,
        n_neighbors= int(normalized_embeddings.shape[0] * (1/32)),
        # min_dist= 0.0, # only influences projection
        metric= "cosine",
    )
umap_embeddings = umap.fit_transform(normalized_embeddings)
# umap_embeddings = umap.fit_transform(embeddings)

In [ ]:
plot_clusters(
    umap_embeddings,
    df_vector_labels.loc[enum_sent_dict.values()],
    'community_SBERT'
    # 'pub_year'
    # 'pmid'
)

plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.show()

#### Graph

In [ ]:
# Create embedding of graph for plotting in 2d.
pos = nx.spring_layout(G, seed=seed) # Takes 12 minutes

In [ ]:
plot_graph(
    G, 
    pos, 
    df_vector_labels.loc[enum_sent_dict.values()],
    'community_SBERT'
)

### Plot only clusters

##### Pre-process (make selection on only clustered sentences)

In [ ]:
# Identify indexes of sentences that are in a community_SBERT cluster
idx_in_cluster = sorted(set(df_vector_labels[df_vector_labels["community_SBERT"] != -1].index))

check1 = df_vector_labels.loc[df_vector_labels["community_SBERT"] != -1, "community_SBERT"].unique()
check1

In [ ]:
# CHECK IF CORRECT
# Select only nodes that are assigned to a cluster
selected_normalized_embeddings = normalized_embeddings.loc[idx_in_cluster]

# This should have the same as print of above
check2 = df_vector_labels.loc[list(idx_in_cluster), "community_SBERT"].unique()
check2

# Is True?
set(check1) == set(check2)

#### PCA

In [ ]:
# Apply PCA
pca = PCA(n_components=2)
pca_embeddings = pca.fit_transform(selected_normalized_embeddings)
# pca_embeddings = pca.fit_transform(embeddings)
pca_embeddings[:5]

In [ ]:
plot_clusters(
    pca_embeddings, 
    df_vector_labels.loc[idx_in_cluster], 
    'community_SBERT'
    # 'pub_year'
)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)")
plt.show()

#### UMAP

In [ ]:
# NOTE: Takes some time... (~1 minute)
umap = UMAP(
        n_components= 2,
        n_neighbors= int(selected_normalized_embeddings.shape[0] * (1/32)),
        # min_dist= 0.0, # only influences projection
        metric= "cosine",
    )
umap_embeddings = umap.fit_transform(selected_normalized_embeddings)
# umap_embeddings = umap.fit_transform(embeddings)

In [ ]:
plot_clusters(
    umap_embeddings,
    df_vector_labels.loc[idx_in_cluster],
    'community_SBERT'
    # 'pub_year'
    # 'pmid'
)

plt.legend()
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.show()

#### Graph

##### Pre-process

In [ ]:
# Back-up G
VOI_G = G.copy()

In [ ]:
# Check process
print(f"Number of nodes before selection: {len(set(VOI_G.nodes))}")

In [ ]:
print(f"Number of sentences that are in a cluster: {len(set(idx_in_cluster))}")

In [ ]:
nodes_to_remove = set(VOI_G.nodes) - set(idx_in_cluster)
print(f"Number of nodes to remove: {len(nodes_to_remove)}")

In [ ]:
# Remove nodes that are not in a cluster
for node_tr in nodes_to_remove:
    VOI_G.remove_node(node_tr)

print(f"New number of nodes: {len(set(VOI_G.nodes))}")

# Check if all went OK
print(f"Only sentences that are in a cluster are now in graph: {set(VOI_G.nodes) == set(idx_in_cluster)}")

##### Continue plotting

In [ ]:
# Create embedding of graph for plotting in 2d.
pos = nx.spring_layout(VOI_G, seed=seed) # Takes 12 minutes

In [ ]:
plot_graph(
    VOI_G, 
    pos, 
    df_vector_labels.loc[idx_in_cluster],
    'community_SBERT'
)